# Imports

In [0]:
from pyspark.sql.functions import *

# Read Bronze Feedback

In [0]:
feedback_bronze_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/bronze/api/customer_feedback.json"

df_feedback = (
    spark.read
    .option("multiline", "true")
    .json(feedback_bronze_path)
)

display(df_feedback)

# Check rows, columns data types

In [0]:
print("Bronze feedback count:", df_feedback.count())
df_feedback.printSchema()

# Clean Feedback

In [0]:
df_feedback_clean = (
    df_feedback
    .dropDuplicates()
    .dropDuplicates(["feedback_id"])
    .filter(col("feedback_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("order_id").isNotNull())
    .withColumn("feedback_id", upper(trim(col("feedback_id"))))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("order_id", col("order_id").cast("int"))
    .withColumn("rating", col("rating").cast("int"))
    .withColumn("comment", trim(col("comment")))
    .withColumn("feedback_date", to_date(col("feedback_date")))
    .filter(col("rating").between(1, 5))
)

# Load Silver Customers and Orders

In [0]:
customers_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/customers/"

df_customers = (
    spark.read
    .format("delta")
    .load(customers_silver_path)
)

orders_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/orders/"

df_orders = (
    spark.read
    .format("delta")
    .load(orders_silver_path)
)

# Validate Customer IDs, Order IDs

In [0]:
invalid_customers = df_feedback_clean.join(
    df_customers,
    on="customer_id",
    how="left_anti"
)

print("Invalid customer references:", invalid_customers.count())

display(invalid_customers)

invalid_orders = df_feedback_clean.join(
    df_orders,
    on="order_id",
    how="left_anti"
)

print("Invalid order references:", invalid_orders.count())

display(invalid_orders)

# Validate Customer + Order relationship

In [0]:
feedback_f = df_feedback_clean.alias("f")
orders_o = df_orders.select(
    "order_id",
    "customer_id"
).alias("o")

invalid_customer_order = (
    feedback_f
    .join(
        orders_o,
        col("f.order_id") == col("o.order_id"),
        "left"
    )
    .filter(
        col("o.order_id").isNull() |
        (col("f.customer_id") != col("o.customer_id"))
    )
)

print(
    "Invalid customer-order relationships:",
    invalid_customer_order.count()
)

display(invalid_customer_order)

# Write Silver Feedback as Delta

In [0]:
feedback_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/customer_feedback/"

(
    df_feedback_clean.write
    .format("delta")
    .mode("overwrite")
    .save(feedback_silver_path)
)

In [0]:
# Verify

In [0]:
df_feedback_silver = (
    spark.read
    .format("delta")
    .load(feedback_silver_path)
)

display(df_feedback_silver)

print(
    "Silver feedback count:",
    df_feedback_silver.count()
)